# 05 — Retrieval Diagnostics

GMM cluster quality, similarity score distributions, cosine heatmaps,
retrieval recall, Camelot wheel, and candidate filtering analysis.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import seaborn as sns
from sklearn.metrics import silhouette_samples, silhouette_score

REPO_ROOT = Path("../..").resolve()
sys.path.insert(0, str(REPO_ROOT / "src"))

from etl.db import get_connection, init_schema
from paths import ARCHIVED_DIR, MODELS_DIR
from recommend.modules.clustering import (
    build_cluster_features,
    filter_corpus_by_cluster,
    predict_cluster_probs,
)
from recommend.modules.similarity import (
    SIMILARITY_FEATURES,
    camelot_distance,
    compute_weighted_cosine,
    find_similar,
    playlist_centroid,
)
from recommend.preprocessing import build_feature_matrix
from utils.logging import configure_logging

configure_logging()
sns.set_theme(style="whitegrid", palette="muted")

conn = get_connection(read_only=False)
init_schema(conn)
corpus = build_feature_matrix(conn)
conn.close()

# Load GMM artifacts
gmm = joblib.load(MODELS_DIR / "gmm_corpus.pkl")
scaler = joblib.load(MODELS_DIR / "scaler_corpus.pkl")
print(f"Corpus: {corpus.shape}, GMM: {gmm.n_components} components")

## 1. GMM Cluster Profiles

In [ ]:
# Assign clusters to full corpus
corpus_features, _ = build_cluster_features(corpus, scaler, fit_scaler=False)
cluster_probs = predict_cluster_probs(corpus_features, gmm)
cluster_ids = np.argmax(cluster_probs, axis=1)

corpus_clustered = corpus.with_columns([
    pl.Series("cluster_id", cluster_ids),
    pl.Series("cluster_prob", cluster_probs.max(axis=1)),
])

# Cluster sizes
cluster_counts = corpus_clustered.group_by("cluster_id").len().sort("cluster_id")
print("Cluster sizes:")
print(cluster_counts)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(cluster_counts["cluster_id"].to_list(), cluster_counts["len"].to_list())
ax.set_xlabel("Cluster ID")
ax.set_ylabel("Track count")
ax.set_title("GMM Cluster Sizes")
plt.tight_layout()
plt.show()

In [ ]:
# Mean audio features per cluster — radar plot
RADAR_FEATURES = [
    "danceability", "energy", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence",
]

cluster_means = (
    corpus_clustered.group_by("cluster_id")
    .agg([pl.col(f).mean() for f in RADAR_FEATURES])
    .sort("cluster_id")
)

angles = np.linspace(0, 2 * np.pi, len(RADAR_FEATURES), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
for row in cluster_means.iter_rows(named=True):
    values = [row[f] for f in RADAR_FEATURES] + [row[RADAR_FEATURES[0]]]
    ax.plot(angles, values, label=f"Cluster {row['cluster_id']}", linewidth=2)
    ax.fill(angles, values, alpha=0.05)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(RADAR_FEATURES, fontsize=9)
ax.set_ylim(0, 1)
ax.legend(loc="upper right", bbox_to_anchor=(1.4, 1.1))
ax.set_title("GMM Cluster Audio Profiles")
plt.tight_layout()
plt.show()

## 2. Silhouette Analysis

In [ ]:
# Subsample for performance (silhouette is O(n^2))
SAMPLE_N = min(5000, len(corpus))
rng = np.random.default_rng(42)
sample_idx = rng.choice(len(corpus), size=SAMPLE_N, replace=False)

X_sample = corpus_features[sample_idx]
labels_sample = cluster_ids[sample_idx]

sil_avg = silhouette_score(X_sample, labels_sample)
sil_values = silhouette_samples(X_sample, labels_sample)
print(f"Overall silhouette score: {sil_avg:.4f}")

# Per-cluster silhouette
fig, ax = plt.subplots(figsize=(10, 8))
y_lower = 0

for i in range(gmm.n_components):
    cluster_sil = np.sort(sil_values[labels_sample == i])
    cluster_size = len(cluster_sil)
    y_upper = y_lower + cluster_size

    ax.fill_betweenx(
        np.arange(y_lower, y_upper),
        0, cluster_sil,
        alpha=0.7, label=f"Cluster {i} (n={cluster_size}, μ={cluster_sil.mean():.3f})",
    )
    y_lower = y_upper + 10

ax.axvline(sil_avg, color="red", linestyle="--", label=f"Mean={sil_avg:.3f}")
ax.set_xlabel("Silhouette coefficient")
ax.set_ylabel("Tracks (sorted within cluster)")
ax.set_title("Silhouette Plot per GMM Cluster")
ax.legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.show()

## 3. Cluster–Genre Alignment

In [ ]:
if "gen_4" in corpus_clustered.columns:
    # Contingency table: cluster_id × gen_4
    contingency = (
        corpus_clustered.filter(pl.col("gen_4").is_not_null())
        .group_by(["cluster_id", "gen_4"])
        .len()
        .pivot(on="gen_4", index="cluster_id", values="len")
        .fill_null(0)
        .sort("cluster_id")
    )

    genre_cols = [c for c in contingency.columns if c != "cluster_id"]
    data = contingency.select(genre_cols).to_numpy()
    # Normalize per cluster (row)
    row_sums = data.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1

    fig, ax = plt.subplots(figsize=(10, 6))
    sns.heatmap(
        data / row_sums,
        xticklabels=genre_cols,
        yticklabels=[f"Cluster {i}" for i in contingency["cluster_id"].to_list()],
        annot=True, fmt=".0%", cmap="YlOrRd", ax=ax,
    )
    ax.set_title("Cluster × gen_4 Distribution (row-normalized)")
    plt.tight_layout()
    plt.show()

## 4. Soft Membership Confidence

In [ ]:
max_probs = cluster_probs.max(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall distribution
axes[0].hist(max_probs, bins=50, edgecolor="white", alpha=0.8)
axes[0].axvline(np.median(max_probs), color="red", linestyle="--", label=f"median={np.median(max_probs):.3f}")
axes[0].set_xlabel("Max cluster probability")
axes[0].set_title("Cluster Confidence Distribution")
axes[0].legend()

# Per-cluster confidence
for i in range(gmm.n_components):
    mask = cluster_ids == i
    axes[1].hist(max_probs[mask], bins=30, alpha=0.5, label=f"Cluster {i}")

axes[1].set_xlabel("Max cluster probability")
axes[1].set_title("Confidence per Cluster")
axes[1].legend(fontsize=7)

plt.tight_layout()
plt.show()

# Stats
print(f"Confidence stats: mean={max_probs.mean():.3f}, median={np.median(max_probs):.3f}, min={max_probs.min():.3f}, max={max_probs.max():.3f}")
print(f"Tracks with prob < 0.5: {(max_probs < 0.5).sum()} ({(max_probs < 0.5).mean()*100:.1f}%)")

## 5. Similarity Score Distributions (per playlist)

In [ ]:
from paths import ARCHIVED_DIR

enoa_path = ARCHIVED_DIR / "enoa.csv"
if enoa_path.exists():
    enoa = pl.read_csv(enoa_path, null_values=["", "NA", "NaN"])
    playlist_names = enoa["playlist_name"].unique().sort().to_list()
else:
    enoa = None
    playlist_names = []

# Analyze similarity for first 3 playlists
PLAYLISTS_TO_ANALYZE = playlist_names[:3]

if enoa is not None and PLAYLISTS_TO_ANALYZE:
    fig, axes = plt.subplots(1, len(PLAYLISTS_TO_ANALYZE), figsize=(6 * len(PLAYLISTS_TO_ANALYZE), 5))
    if len(PLAYLISTS_TO_ANALYZE) == 1:
        axes = [axes]

    corpus_ids = corpus["id"].cast(pl.Utf8).to_list()

    for idx, pname in enumerate(PLAYLISTS_TO_ANALYZE):
        pids = set(
            enoa.filter(pl.col("playlist_name") == pname)["id"].cast(pl.Utf8).to_list()
        )
        labels = np.array([1 if tid in pids else 0 for tid in corpus_ids])

        # Compute similarity to playlist centroid
        pos_mask = labels == 1
        if pos_mask.sum() < 5:
            axes[idx].set_title(f"{pname} (too few positives)")
            continue

        centroid = playlist_centroid(
            corpus.filter(pl.Series(pos_mask)),
            SIMILARITY_FEATURES,
        )
        X = corpus.select(SIMILARITY_FEATURES).to_numpy().astype(np.float64)
        weights = np.ones(len(SIMILARITY_FEATURES))
        scores = compute_weighted_cosine(X, centroid, weights)

        axes[idx].hist(scores[labels == 0], bins=50, alpha=0.6, label="Negative", density=True)
        axes[idx].hist(scores[labels == 1], bins=30, alpha=0.7, label="Positive", density=True)
        axes[idx].set_xlabel("Similarity score")
        axes[idx].set_title(f"{pname} (n_pos={pos_mask.sum()})")
        axes[idx].legend()

    plt.suptitle("Similarity Score: Positives vs Negatives", fontsize=13)
    plt.tight_layout()
    plt.show()

## 6. Cosine Similarity Heatmap (within a playlist)

In [ ]:
HEATMAP_PLAYLIST = PLAYLISTS_TO_ANALYZE[0] if PLAYLISTS_TO_ANALYZE else None

if HEATMAP_PLAYLIST and enoa is not None:
    pids = set(
        enoa.filter(pl.col("playlist_name") == HEATMAP_PLAYLIST)["id"].cast(pl.Utf8).to_list()
    )
    corpus_ids_list = corpus["id"].cast(pl.Utf8).to_list()
    pos_mask = pl.Series([tid in pids for tid in corpus_ids_list])
    playlist_tracks = corpus.filter(pos_mask)

    # Limit to 50 tracks for readability
    if len(playlist_tracks) > 50:
        playlist_tracks = playlist_tracks.head(50)

    X_pl = playlist_tracks.select(SIMILARITY_FEATURES).to_numpy().astype(np.float64)
    # Pairwise cosine similarity
    from sklearn.metrics.pairwise import cosine_similarity
    sim_matrix = cosine_similarity(X_pl)

    track_labels = playlist_tracks["track_name"].to_list() if "track_name" in playlist_tracks.columns else [f"t{i}" for i in range(len(playlist_tracks))]

    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(
        sim_matrix,
        xticklabels=track_labels, yticklabels=track_labels,
        cmap="YlOrRd", vmin=0, vmax=1, ax=ax,
    )
    ax.set_title(f"Track-to-Track Cosine Similarity — {HEATMAP_PLAYLIST}")
    plt.xticks(rotation=90, fontsize=6)
    plt.yticks(fontsize=6)
    plt.tight_layout()
    plt.show()

## 7. Retrieval Recall

In [ ]:
# For each playlist: run find_similar(centroid, k=100), measure recall of actual playlist tracks
if enoa is not None and PLAYLISTS_TO_ANALYZE:
    recall_results = []

    for pname in playlist_names:
        pids = set(
            enoa.filter(pl.col("playlist_name") == pname)["id"].cast(pl.Utf8).to_list()
        )
        corpus_pids = pids & set(corpus["id"].cast(pl.Utf8).to_list())
        if len(corpus_pids) < 10:
            continue

        pos_mask = pl.Series([tid in corpus_pids for tid in corpus["id"].cast(pl.Utf8).to_list()])
        positives_df = corpus.filter(pos_mask)
        centroid = playlist_centroid(positives_df, SIMILARITY_FEATURES)

        # Build query dict from centroid
        query_dict = {feat: float(centroid[i]) for i, feat in enumerate(SIMILARITY_FEATURES)}

        results = find_similar(corpus, query_dict, k=100, exclude_ids=set())
        retrieved_ids = set(results["id"].cast(pl.Utf8).to_list()) if "id" in results.columns else set()
        hits = len(retrieved_ids & corpus_pids)
        recall = hits / len(corpus_pids) if corpus_pids else 0

        recall_results.append({"playlist": pname, "n_pos": len(corpus_pids), "hits_at_100": hits, "recall_at_100": round(recall, 3)})

    recall_df = pl.DataFrame(recall_results).sort("recall_at_100", descending=True)
    print("Retrieval Recall@100 (cosine from centroid):")
    display(recall_df)

    fig, ax = plt.subplots(figsize=(10, max(4, len(recall_df) * 0.4)))
    ax.barh(recall_df["playlist"].to_list()[::-1], recall_df["recall_at_100"].to_list()[::-1])
    ax.set_xlabel("Recall@100")
    ax.set_title("Retrieval Recall@100 per Playlist")
    ax.axvline(0.5, color="red", linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()

## 8. Camelot Wheel Analysis

In [ ]:
if "key_mode" in corpus.columns and HEATMAP_PLAYLIST and enoa is not None:
    pids = set(
        enoa.filter(pl.col("playlist_name") == HEATMAP_PLAYLIST)["id"].cast(pl.Utf8).to_list()
    )
    pos_mask = pl.Series([tid in pids for tid in corpus["id"].cast(pl.Utf8).to_list()])
    playlist_df = corpus.filter(pos_mask)

    key_modes = playlist_df["key_mode"].to_list()
    # Modal key
    modal_key = max(set(key_modes), key=key_modes.count) if key_modes else ""
    distances = [camelot_distance(km, modal_key) for km in key_modes]

    print(f"Playlist: {HEATMAP_PLAYLIST}")
    print(f"Modal key: {modal_key}")
    print(f"Mean Camelot distance: {np.mean(distances):.2f}")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Key distribution
    from collections import Counter
    key_counts = Counter(key_modes)
    keys_sorted = sorted(key_counts.keys())
    axes[0].bar(keys_sorted, [key_counts[k] for k in keys_sorted])
    axes[0].set_xlabel("Key-Mode")
    axes[0].set_title(f"Key Distribution — {HEATMAP_PLAYLIST}")
    axes[0].tick_params(axis="x", rotation=90)

    # Camelot distance distribution
    axes[1].hist(distances, bins=range(8), edgecolor="white", alpha=0.8, align="left")
    axes[1].set_xlabel("Camelot Distance from Modal Key")
    axes[1].set_title(f"Camelot Distance Distribution (modal={modal_key})")
    axes[1].set_xticks(range(7))

    plt.tight_layout()
    plt.show()

## 9. Candidate Filtering Funnel

In [ ]:
# For a query track, show: full corpus → cluster filter → cosine top-100
if len(corpus) > 0:
    query_row = corpus.row(0, named=True)
    query_name = query_row.get("track_name", "row_0")
    print(f"Query: {query_name}")

    # Step 1: Full corpus
    n_full = len(corpus)

    # Step 2: Cluster filter
    query_features, _ = build_cluster_features(corpus.head(1), scaler, fit_scaler=False)
    query_probs = predict_cluster_probs(query_features, gmm)
    filtered = filter_corpus_by_cluster(corpus, query_probs[0], gmm, scaler, min_prob=0.05)
    n_cluster = len(filtered)

    # Step 3: Cosine top-100
    results = find_similar(filtered, query_row, k=100, exclude_ids={query_row.get("id", "")})
    n_cosine = len(results)

    print(f"\nFunnel:")
    print(f"  Full corpus:     {n_full:,}")
    print(f"  After cluster:   {n_cluster:,} ({n_cluster/n_full*100:.1f}%)")
    print(f"  After cosine:    {n_cosine} (top-k)")

    # Visualize as funnel
    fig, ax = plt.subplots(figsize=(8, 4))
    stages = ["Full corpus", "Cluster filter", "Cosine top-100"]
    counts = [n_full, n_cluster, n_cosine]
    ax.barh(stages[::-1], counts[::-1], color=["#3498db", "#e74c3c", "#2ecc71"])
    for i, (stage, count) in enumerate(zip(stages[::-1], counts[::-1])):
        ax.text(count + n_full * 0.02, i, f"{count:,}", va="center")
    ax.set_xlabel("Candidates")
    ax.set_title(f"Retrieval Funnel — query: {query_name}")
    plt.tight_layout()
    plt.show()